# Calculating the Big Four
### Dataset: from sklearn.datasets import fetch_california_housing
##### The Task: Train a standard regression model on a dataset of your choice.
## Objectives:
#### Calculate the Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared (R^2) scores using sklearn.metrics.
#### Write a short summary explaining what each metric tells you about your model's performance in plain English.

In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load the dataset
california = fetch_california_housing()
X, y = california.data, california.target

# 2. Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train a standard regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Make predictions
y_pred = model.predict(X_test)

# 5. Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("--- California Housing Model Performance ---")
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R-squared: {r2:.4f}")

--- California Housing Model Performance ---
MAE: 0.5332
MSE: 0.5559
RMSE: 0.7456
R-squared: 0.5758


What These Metrics Tell You 
- Mean Absolute Error (MAE): This is the average of all the absolute errors. In plain English, it tells you on average how far off your model's predictions are from the actual values, treating all errors equally regardless of direction or size.
- Mean Squared Error (MSE): This calculates the average of the squared differences between predicted and actual values. Because the errors are squared before being averaged, MSE severely penalizes larger errors. It tells you if your model is occasionally making massive mistakes, but the number is hard to interpret since it's in "squared" units.
- Root Mean Squared Error (RMSE): This is simply the square root of the MSE. It brings the error metric back into the original units of your target variable (like dollars or test scores), making it easier to read while still keeping the heavy penalty for large errors.
- R-squared ($R^2$): This represents the proportion of the variance in the dependent variable that is predictable from the independent variables. Essentially, it tells you how much better your model is at predicting the target compared to a model that just guesses the average every time. A score of 1.0 is perfect, and 0.0 means your model is no better than taking a simple average.

# The Impact of Outliers
### Dataset: study_hour.csv
#### The Task: Investigate how different metrics handle extreme errors.
## Objectives:
#### Take a clean dataset, train a model, and record the MAE and RMSE.
#### Artificially inject a few massive outliers into your target variable (y).
#### Retrain the model and recalculate the MAE and RMSE.
#### Observe and explain why the RMSE jumps drastically compared to the MAE, demonstrating MSE/RMSE's sensitivity to outliers.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('study_hour.csv')

# Features and target
X = df[['Hours_Studied']]
y = df['Test_Score']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Clean Dataset Model
model_clean = LinearRegression()
model_clean.fit(X_train, y_train)
y_pred_clean = model_clean.predict(X_test)

mae_clean = mean_absolute_error(y_test, y_pred_clean)
rmse_clean = np.sqrt(mean_squared_error(y_test, y_pred_clean))

print("--- Clean Dataset ---")
print(f"MAE:  {mae_clean:.4f}")
print(f"RMSE: {rmse_clean:.4f}")

# --- Inject Massive Outliers ---
# We make a copy of our training targets and artificially make the first 3 scores massive
y_train_outliers = y_train.copy()
outlier_indices = y_train_outliers.index[:3]
y_train_outliers.loc[outlier_indices] = [5000, 10000, 15000]

# --- Retrain with Outliers ---
model_outlier = LinearRegression()
model_outlier.fit(X_train, y_train_outliers)
y_pred_outlier = model_outlier.predict(X_test)

mae_outlier = mean_absolute_error(y_test, y_pred_outlier)
rmse_outlier = np.sqrt(mean_squared_error(y_test, y_pred_outlier))

print("\n--- Dataset with Outliers ---")
print(f"MAE:  {mae_outlier:.4f}")
print(f"RMSE: {rmse_outlier:.4f}")

--- Clean Dataset ---
MAE:  16.8803
RMSE: 20.6143

--- Dataset with Outliers ---
MAE:  181.9793
RMSE: 185.7475


Observation & Explanation
When ran this analysis on the file, the baseline clean data produced a MAE of ~16.88 and an RMSE of ~20.61.

When massive outliers were injected (scores inflated to 5000+), both errors naturally increased because the model was dragged off course. However, RMSE is mathematically designed to be far more sensitive to large errors.

Because MSE (and by extension, RMSE) squares the difference between the prediction and the actual value, a small error of 2 becomes an penalty of 4, but a large error of 100 becomes a massive penalty of 10,000. MAE, on the other hand, just takes the absolute value, treating an error of 100 exactly linearly as 50 times worse than an error of 2. Consequently, when outliers are present, they pull the RMSE up disproportionately compared to the MAE, making RMSE an excellent metric to track if large errors are strictly unacceptable for your specific problem.